# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2'
openai = OpenAI()

API key looks good so far


In [12]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [5]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [6]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [7]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'career/linkedin profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'career/twitter profile',
   'url': 'https://twitter.com/edwarddonner'},
  {'type': 'career/facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'courses curriculum',
   'url': 'https://edwarddonner.com/curriculum/'}]}

In [8]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [9]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company/mission', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company/mission', 'url': 'https://edwarddonner.com/'}]}

In [10]:
select_relevant_links("https://visionxacademy.vercel.app/")

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 7 relevant links


{'links': [{'type': 'company page',
   'url': 'https://visionxacademy.vercel.app/'},
  {'type': 'careers/jobs page', 'url': 'https://visionxacademy.vercel.app/'},
  {'type': 'reviews google', 'url': 'https://visionxacademy.vercel.app/'},
  {'type': 'maps google', 'url': 'https://visionxacademy.vercel.app/'},
  {'type': 'google driving instructions',
   'url': 'https://www.google.com/maps/place/VisionX+Academy,+Macarthur/data=!4m2!3m1!1s0x80a3f6a7207da12b:0x9ae972a497994647'},
  {'type': 'book a lesson',
   'url': "https://wa.me/61481770888?text=Hi%20Binod%2C%20I'd%20like%20to%20book%20a%20driving%20lesson%20with%20VisionX%20Academy."},
  {'type': 'book a lesson phone', 'url': 'tel:+61481770888'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://visionxacademy.vercel.app/"))

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 5 relevant links
## Landing Page:

VisionX Academy | Driving Lessons in Macarthur, Sydney

Academy
Why us
Lessons
NSW rules
Reviews
Contact
+61 481 770 888
Macarthur · Camden · Campbelltown
Learn to drive with
confidence
, not just to pass the test.
VisionX Academy is a top-rated local driving school in Sydney's south-west. Friendly 1-on-1 lessons with instructor
Binod Poudel
that turn nervous beginners into safe, road-ready drivers.
Call
+61 481 770 888
Book on WhatsApp
5.0
on Google ·
14
reviews
Open 7 days from 6 AM
50
L
Built-up area · 50 km/h
Why learn with VisionX
A nicer way to learn to drive
Learning to drive should feel exciting, not stressful. Here's why Macarthur learners and their parents choose us.
1-on-1, never rushed
Every lesson is just you and Binod — paced around how you learn, not a checklist.
Safety over shortcuts
We build real roadcraft and lifelong habits, not just enough to 

In [22]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
Important:
1. The brochure body content should be written in Nepali language.
2. All headings (H1, H2, H3, etc.) must be written in English only — do NOT translate them.
3. Keep company names, brand names, and website URLs exactly as they appear — do not translate or transliterate them.
4. Use simple and clear Nepali language for body text that is easy to understand for a wide audience.
"""

In [23]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages.
Use this information to build a short brochure of the company in markdown without code blocks.

Rules:
- Write all body content in Nepali.
- Write all headings (H1, H2, H3) in English only — do NOT translate headings.
- The company name "{company_name}" must appear exactly as written above — never translate or transliterate it.
- Do not translate any brand names, product names, or URLs into Nepali.

"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt

In [24]:
get_brochure_user_prompt("visionxacademy", "https://visionxacademy.vercel.app/")

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 14 relevant links


'\nYou are looking at a company called: visionxacademy\nHere are the contents of its landing page and other relevant pages.\nUse this information to build a short brochure of the company in markdown without code blocks.\n\nRules:\n- Write all body content in Nepali.\n- Write all headings (H1, H2, H3) in English only — do NOT translate headings.\n- The company name "visionxacademy" must appear exactly as written above — never translate or transliterate it.\n- Do not translate any brand names, product names, or URLs into Nepali.\n\n## Landing Page:\n\nVisionX Academy | Driving Lessons in Macarthur, Sydney\n\nAcademy\nWhy us\nLessons\nNSW rules\nReviews\nContact\n+61 481 770 888\nMacarthur · Camden · Campbelltown\nLearn to drive with\nconfidence\n, not just to pass the test.\nVisionX Academy is a top-rated local driving school in Sydney\'s south-west. Friendly 1-on-1 lessons with instructor\nBinod Poudel\nthat turn nervous beginners into safe, road-ready drivers.\nCall\n+61 481 770 888\nB

In [25]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("visionxacademy", "https://visionxacademy.vercel.app/")

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 5 relevant links


**VisionX Academy**
**Driving Lessons in Macarthur, Sydney**

**About Us**
VisionX Academy ही Sydneюयस दक्षिण-पश्चिम भागका लागि एक प्रशंसित स्थानीय ड्राइving स्कूल भन्छ्छे। ड्राइंग सेमिनार एक द्वितीयक कक्षा है जो ड्राइभर्स को रोड-रेति रणनीति बनाने के लिए लचीला प्रशिक्षण प्रदान कर्छे की भौतिक दृष्टिकोण से प्रयास गर्छे। बिनोद पौडेल यक्त 1-1 ड्राइभिंग लesson गर्छ्छे जो परिचित शिक्षकsका युवा ड्राइकरsको रोड-रेति कौशल सम्पन्न कर्छे।

**Features**

*   नशीला ड्राइभिंग सेमिनार (Driving Lesson)
*   एक प्रशंसित स्थानीय ड्राइंग स्कूल (local driving school)
*   1-1 ड्राइभिंग लesson (1-on-1 driving lesson)
*   ड्राइभिंग सेमिनार भौतिक रूपसे प्रयास गर्छे (practical driving lessons)
*   रोड-रेति कौशल बनाना (building road-craft habits)
*   नशीला ड्राइभर बनाना ( becoming a safe driver)
*   ड्राइभिंग सेमिनार के लिए लचीला प्रशिक्षण (structured prep for NSW driving test)
*   ड्राइभर को रोड-रेति कौशल सम्पन्न बनाना (making students skilled drivers)

**What Our Students Say**

*   ' Binod Paudel yahi ek bahut hi santulit aur prashansak instructor hai jo kisi bhi level ke driver ko confident banane mein madad kartaa hai.'

**How to Book a Lesson**

*   Call : +61 481 770 888
*   Book On WhatsApp
*   Contact Us

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [29]:
stream_brochure("visionxacademy", "https://visionxacademy.vercel.app/")

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 1 relevant links


# VisionX Academy

## About Us

VisionX Academy is a top-rated local driving school in Sydney's south-west, committed to delivering safe and confident learning experiences. Our driving school is built on the principles of personalized attention, calm and patient teaching, and a focus on lifelong driving habits.

#### Our Mission

We aim to empower our students with the skills and confidence to become safe and independent drivers, not just to pass the test. With every lesson, we strive to create a nurturing and supportive environment that fosters a love for driving, rather than just a sense of duty.

#### Why Choose Us?

* 1-on-1 lessons with instructor Binod Poudel, tailored to your learning style and pace
* Safety over shortcuts: we focus on building real roadcraft and lifelong habits, not just passing the test
* Calm and patient teaching: no judgement, every time
* Local Macarthur roads: practice on the streets, roundabouts, and test routes you'll actually drive on
* Early and late sessions: lessons from 6 AM, seven days a week, to fit around your schedule

#### Our Services

* L-License Lessons: everything you need to get started
* P-License Lessons: take your driving to the next level with confidence building and test preparation
* Car Control & Observation: mastering roundabouts, merging, and lane changes
* Regain Your Nerve: confidence building for anxious drivers
* Overseas License Holders: learn to drive in Australia with ease

#### Join Our Community

Visit us today and experience the difference of learning to drive with confidence, not just to pass the test. Contact us at +61 481 770 888 or book your lesson on WhatsApp.

#### Location

Macarthur, Sydney - Camden, Campbelltown

#### Follow Us

Stay up to date with the latest news, promotions, and events from VisionX Academy on our social media channels:

* Google Maps
* Facebook
* Instagram

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>